# Experiment 1: held-out benchmark — LGD

This notebook evaluates the full predefined sweep without reducing it to a post-hoc winner. Every adapted recipe is paired with its own untuned base on the same dataset and outer folds. Five outer folds are averaged within a dataset; datasets then receive equal weight.

PD reports AUC change and complementary probability metrics. LGD reports fractional RMSE reduction within each paired fold before dataset averaging. Complete outer-fold coverage is required for each displayed model–dataset effect. Missing controls and failed folds are not silently replaced. These descriptive results are conditional on the 25-table corpus and its dependent partitions.

In [1]:
%matplotlib inline
import sys
from pathlib import Path
REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from IPython.display import display
from src.visualize import style
from src.visualize.figures import FigureSaver
from src.visualize import campaign as cp
from src.data.dataset_names import display_frame
from src.visualize.inputs import analysis_root
style.apply()
sink = FigureSaver('experiment1/04_results_lgd')
report = cp.NotebookReport('Experiment 1: held-out benchmark — LGD')
TRACK = 'lgd'


## 1. Benchmark availability and coverage

Training completion is separate from evaluation completion. A dataset effect enters the analysis only when both the adapted model and its untuned reference have every configured outer fold.

In [2]:
run = cp.load_campaign(1, TRACK)
cp.show(sink, cp.plot_eval_coverage(run))
benchmark = cp.benchmark_effects(run)
report.add('1. Benchmark coverage', cp.coverage_summary(run)+'\n'+str(len(benchmark))+' complete paired recipe–dataset effects.')

No measurements available for this section yet.


## 2. The response surface

Learning rate and L2-SP remain separate axes; adaptation modes and bases remain separate figures. The displayed endpoint was configured in advance.

In [3]:
cp.show(sink, cp.plot_response_surfaces(benchmark, run.metric))
report.add('2. Paired response surfaces', cp.effect_summary(benchmark))

No measurements available for this section yet.


## 3. Matched effects of anchoring and adaptation

These contrasts hold the other recipe factors and dataset fixed. Dataset points show heterogeneous behavior rather than treating folds or hundreds of recipes as independent replications.

In [4]:
anchor = cp.factor_contrasts(benchmark, 'l2sp_lambda', 0.0, 0.003)
adaptation = cp.factor_contrasts(benchmark, 'frozen', False, True)
cp.show(sink, cp.plot_contrasts(anchor, run.metric, 'L2-SP 0.003 minus 0'))
cp.show(sink, cp.plot_contrasts(adaptation, run.metric, 'Frozen minus full'))
report.add('3. Matched factor contrasts', 'L2-SP:\n'+cp.effect_summary(anchor,'contrast')+'\nAdaptation:\n'+cp.effect_summary(adaptation,'contrast'))

No measurements available for this section yet.
No measurements available for this section yet.


## 4. Dataset-level heterogeneity

Each page contains a bounded slice of the full matrix, with the same scale across pages for a fixed base/adaptation. Empty cells remain empty; no top-k recipe filter hides adverse effects.

In [5]:
cp.show(sink, cp.plot_dataset_pages(benchmark, run.metric))
report.add('4. Dataset-level effects', benchmark.groupby('dataset').effect.agg(['count','median','min','max']).to_string() if not benchmark.empty else 'No complete paired effects.')

No measurements available for this section yet.


## 5. Complementary metrics and trade-offs

PD compares AUC effects with Brier-score effects, which mix discrimination and calibration. LGD compares relative RMSE effects with absolute MAE improvements. The panels do not treat those units as interchangeable.

In [6]:
cp.show(sink, cp.plot_secondary_tradeoff(run))
secondary = cp.benchmark_effects(run, 'brier_score' if TRACK == 'pd' else 'mae')
report.add('5. Secondary metrics', cp.effect_summary(secondary))

No measurements available for this section yet.


## 6. Context from untuned models and classical controls

Keep the baseline landscape separate from the adapted grid. Within-dataset ranks avoid pooling raw LGD scales; they discard effect magnitude, so the paired-effect figures remain the primary analysis.

In [7]:
cp.show(sink, cp.plot_control_context(run))
report.add('6. Reference-model context', 'Ranks require complete outer folds and all displayed controls for a dataset. They describe reference performance, not an independently selected winner.')

No measurements available for this section yet.


## 7. Training cost versus effect

Plot recipe-level effects against observed training GPU allocation time. This is not inference latency and excludes classical HPO/evaluation cost. Missing timing measurements are not zero-cost points.

In [8]:
cp.show(sink, cp.plot_cost_effect(run))
report.add('7. Cost versus effect', 'One point per measured recipe; dataset effects are equally weighted, and training allocation hours are summarized across completed partitions.')

No measurements available for this section yet.


## 8. Scope of the evidence

The sweep describes how adaptation behaves across knobs and datasets. It cannot establish universal superiority, seed robustness everywhere, absence of pretraining contamination, temporal deployment performance, or retention outside credit data.

In [9]:
report.add('8. Interpretation limits', 'Descriptive corpus-conditional comparison. No held-out winner selection, no fold-as-dataset independence assumption, and no non-credit forgetting measurement.')

## Summary

The following text repeats the sections in order. It is included verbatim in `All_Results.md`.

In [10]:
print(report.summary(sink))

Experiment 1: held-out benchmark — LGD

1. Benchmark coverage
Run cpt_main_v4; LGD; 256 planned trials; target 5,000 successful updates.
status
PENDING    256
0 trajectory records; 0 benchmark fold records.
0 complete paired recipe–dataset effects.

2. Paired response surfaces
No matched measurements available; absence is not a zero effect.

3. Matched factor contrasts
L2-SP:
No matched measurements available; absence is not a zero effect.
Adaptation:
No matched measurements available; absence is not a zero effect.

4. Dataset-level effects
No complete paired effects.

5. Secondary metrics
No matched measurements available; absence is not a zero effect.

6. Reference-model context
Ranks require complete outer folds and all displayed controls for a dataset. They describe reference performance, not an independently selected winner.

7. Cost versus effect
One point per measured recipe; dataset effects are equally weighted, and training allocation hours are summarized across completed part